In [7]:
# 1. Mount Google Drive
from google.colab import drive
import json
import numpy as np
import pandas as pd

# This will prompt you to log in and give Colab access to your Drive
drive.mount('/content/drive')

# Define the path to your Drive folder
BASE_PATH = "/content/drive/MyDrive/PDC_Project_M1/"

def standardize_path(path_str):
    """
    Forces all slashes to be the same, then splits the string to get JUST the filename.
    This bypasses the Windows vs Linux path issue entirely.
    """
    if pd.isna(path_str) or not str(path_str).strip():
        return "invalid_path"

    # 1. Convert it to a string
    clean_str = str(path_str)
    # 2. Replace all Windows backslashes with Linux forward slashes
    clean_str = clean_str.replace('\\', '/')
    # 3. Split by the slash and take the very last item (the filename)
    filename = clean_str.split('/')[-1]

    return filename

def align_datasets():
    print("Loading data from Google Drive...")

    # 1. Load Raviha's Hashes
    with open(f'{BASE_PATH}hash_results.json', 'r') as f:
        raviha_data = json.load(f)
    raviha_df = pd.DataFrame.from_dict(raviha_data, orient='index')
    raviha_df.index = raviha_df.index.map(standardize_path)
    raviha_df.index.name = 'image_id'

    # 2. Load Fatima's Traditional Features Map
    fatima_paths = np.load(f'{BASE_PATH}image_paths.npy', allow_pickle=True)
    fatima_df = pd.DataFrame({'fatima_row_index': range(len(fatima_paths))})
    fatima_df['image_id'] = [standardize_path(p) for p in fatima_paths]
    fatima_df.set_index('image_id', inplace=True)

    # 3. Load Arham's Deep Features Metadata
    arham_metadata = pd.read_csv(f'{BASE_PATH}metadata.csv')
    arham_df = pd.DataFrame({'arham_row_index': range(len(arham_metadata))})
    arham_df['image_id'] = arham_metadata['image_path'].apply(standardize_path)
    arham_df.set_index('image_id', inplace=True)

    print("Aligning datasets...")
    # 4. Merge everything together based on the 'image_id'
    aligned_data = raviha_df.join([fatima_df, arham_df], how='inner')

    if "invalid_path" in aligned_data.index:
        aligned_data = aligned_data.drop(index="invalid_path")

    print(f"\n✅ Successfully aligned {len(aligned_data)} images!")
    print("\nSample of the aligned master map:")
    display(aligned_data.head())

    aligned_data.to_csv(f'{BASE_PATH}master_alignment_map.csv')
    print(f"\nSaved master map to {BASE_PATH}master_alignment_map.csv")

    return aligned_data

# Run the alignment
master_map = align_datasets()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data from Google Drive...
Aligning datasets...

✅ Successfully aligned 7128 images!

Sample of the aligned master map:


,aHash,dHash,pHash,fatima_row_index,arham_row_index
image_id,,,,,
2696.jpg,ffffe9fcf8f8fc00,c4cadbc933b0f090,c1a317d62e866b99,4,4
2694.jpg,0e0e0f0206eef6fe,bc3c7a7e5cc8e8e2,951dd0722d67c1c7,2,2
2692.jpg,3c3c3e7e5e9e0c00,e9f8e4d4b43458c9,96856bfa61676e08,0,0
2693.jpg,1f1f1f1f3f2c0000,bef8f2b6e8c986a5,979d72939e24528b,1,1
2699.jpg,8081133e1d78f87e,2827a6b8f9f19198,886e775b13d1862e,7,7



Saved master map to /content/drive/MyDrive/PDC_Project_M1/master_alignment_map.csv


In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

BASE_PATH = "/content/drive/MyDrive/PDC_Project_M1/"

def fuse_features():
    print("Loading mapped data and raw feature matrices...")

    # 1. Load the master map we just created
    # We use image_id as the index so we know exactly which image is which
    master_map = pd.read_csv(f'{BASE_PATH}master_alignment_map.csv', index_col='image_id')

    # 2. Load the raw feature matrices
    fatima_features_raw = np.load(f'{BASE_PATH}fatima_features.npy')
    arham_features_raw = np.load(f'{BASE_PATH}arham_features.npy')

    print(f"Fatima's raw matrix shape: {fatima_features_raw.shape}")
    print(f"Arham's raw matrix shape: {arham_features_raw.shape}")

    # 3. Extract the features in the EXACT order of our master_map
    # This guarantees row 0 in our new matrix is row 0 in the master_map
    fatima_indices = master_map['fatima_row_index'].values
    arham_indices = master_map['arham_row_index'].values

    fatima_aligned = fatima_features_raw[fatima_indices]
    arham_aligned = arham_features_raw[arham_indices]

    print("\nNormalizing feature spaces...")
    # 4. L2 Normalization (Forces all vectors to have a magnitude of 1)
    # This ensures Deep Features and Traditional Features have equal "weight"
    fatima_norm = normalize(fatima_aligned, norm='l2')
    arham_norm = normalize(arham_aligned, norm='l2')

    print("Fusing features via concatenation...")
    # 5. Concatenate them horizontally (stitch them side-by-side)
    # If Fatima has 500 features and Arham has 2048, the fused vector will be 2548
    fused_features = np.hstack((fatima_norm, arham_norm))

    print(f"\n✅ Final Fused Matrix Shape: {fused_features.shape}")

    # 6. Save the final fused matrix
    np.save(f'{BASE_PATH}fused_features.npy', fused_features)
    print(f"Saved fused features to {BASE_PATH}fused_features.npy")

    return fused_features

# Run the fusion pipeline
fused_matrix = fuse_features()

Loading mapped data and raw feature matrices...
Fatima's raw matrix shape: (6378, 672)
Arham's raw matrix shape: (6378, 512)

Normalizing feature spaces...
Fusing features via concatenation...

✅ Final Fused Matrix Shape: (7128, 1184)
Saved fused features to /content/drive/MyDrive/PDC_Project_M1/fused_features.npy


In [9]:
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

BASE_PATH = "/content/drive/MyDrive/PDC_Project_M1/"

def standardize_path_perfect(path_str):
    """ Fixes the slash issue BUT keeps the folder name to ensure uniqueness """
    if pd.isna(path_str) or not str(path_str).strip(): return "invalid"
    parts = str(path_str).replace('\\', '/').split('/')
    # Returns "folder_name/image_name.jpg"
    return f"{parts[-2]}/{parts[-1]}" if len(parts) >= 2 else parts[-1]

print("1. Building the Perfected Database...")
# Load and map correctly
with open(f'{BASE_PATH}hash_results.json', 'r') as f:
    raviha_df = pd.DataFrame.from_dict(json.load(f), orient='index')
raviha_df.index = raviha_df.index.map(standardize_path_perfect)

fatima_paths = np.load(f'{BASE_PATH}image_paths.npy', allow_pickle=True)
fatima_df = pd.DataFrame({'fatima_idx': range(len(fatima_paths))})
fatima_df.index = [standardize_path_perfect(p) for p in fatima_paths]

arham_meta = pd.read_csv(f'{BASE_PATH}metadata.csv')
arham_df = pd.DataFrame({'arham_idx': range(len(arham_meta))})
arham_df.index = arham_meta['image_path'].apply(standardize_path_perfect)

# Merge and drop any accidental exact duplicates
master_map = raviha_df.join([fatima_df, arham_df], how='inner')
master_map = master_map[~master_map.index.duplicated(keep='first')]

# Fuse Math
f_raw = np.load(f'{BASE_PATH}fatima_features.npy')
a_raw = np.load(f'{BASE_PATH}arham_features.npy')
f_norm = normalize(f_raw[master_map['fatima_idx'].values], norm='l2')
a_norm = normalize(a_raw[master_map['arham_idx'].values], norm='l2')
fused_matrix = np.hstack((f_norm, a_norm))

print(f"✅ Corrected Map Shape: {master_map.shape[0]} images.")
print(f"✅ Corrected Matrix Shape: {fused_matrix.shape}")

# ==========================================
# PHASE 3: THE REVERSE IMAGE SEARCH ENGINE
# ==========================================

def search_similar_images(query_image_id, top_k=5):
    print(f"\n🔍 Searching database for matches to: '{query_image_id}'...")

    # 1. Find the math vector for our query image
    query_idx = master_map.index.get_loc(query_image_id)
    query_vector = fused_matrix[query_idx].reshape(1, -1)

    # 2. Compute Cosine Similarity against the ENTIRE database simultaneously
    # A score of 1.0 means identical, 0.0 means completely different
    similarities = cosine_similarity(query_vector, fused_matrix)[0]

    # 3. Get the indices of the top results (sorting them highest to lowest)
    top_indices = similarities.argsort()[-(top_k+1):][::-1]

    # 4. Remove the query image from its own results (it will obviously have a 1.0 score)
    top_indices = [i for i in top_indices if i != query_idx][:top_k]

    print(f"\n--- Top {top_k} Results ---")
    for rank, idx in enumerate(top_indices):
        match_id = master_map.index[idx]
        score = similarities[idx]

        # We check if the match is in the same folder (e.g., 'dew') as the query
        query_folder = query_image_id.split('/')[0]
        match_folder = match_id.split('/')[0]
        is_correct = "✅" if query_folder == match_folder else "❌"

        print(f"{rank+1}. {match_id} (Similarity: {score:.4f}) {is_correct}")

# Let's test the engine on the very first image in our database!
test_query = master_map.index[0]
search_similar_images(test_query, top_k=5)

1. Building the Perfected Database...
✅ Corrected Map Shape: 6378 images.
✅ Corrected Matrix Shape: (6378, 1184)

🔍 Searching database for matches to: 'dew/2696.jpg'...

--- Top 5 Results ---
1. dew/2841.jpg (Similarity: 0.8610) ✅
2. dew/2758.jpg (Similarity: 0.8442) ✅
3. glaze/6115.jpg (Similarity: 0.8389) ❌
4. dew/2708.jpg (Similarity: 0.8353) ✅
5. dew/2789.jpg (Similarity: 0.8317) ✅


In [10]:
def evaluate_baseline(k=5, num_test_queries=100):
    print(f"📊 Running Baseline Evaluation (k={k}, queries={num_test_queries})...")

    # 1. Figure out how many images exist per category (needed for Recall)
    # This counts how many 'dew', 'glaze', etc., exist in the master map
    folders = [idx.split('/')[0] for idx in master_map.index]
    category_counts = pd.Series(folders).value_counts().to_dict()

    # 2. Pick a random subset of queries for our test
    np.random.seed(42) # Keeps the random selection consistent
    test_indices = np.random.choice(len(master_map), num_test_queries, replace=False)
    test_queries = master_map.index[test_indices]

    total_precision = 0
    total_recall = 0

    for query_id in test_queries:
        query_folder = query_id.split('/')[0]
        # Total valid targets (excluding the query image itself)
        total_relevant_in_db = category_counts[query_folder] - 1

        # Run similarity
        query_idx = master_map.index.get_loc(query_id)
        query_vector = fused_matrix[query_idx].reshape(1, -1)
        similarities = cosine_similarity(query_vector, fused_matrix)[0]

        # Get top K indices (excluding the query itself)
        top_indices = similarities.argsort()[-(k+1):][::-1]
        top_indices = [i for i in top_indices if i != query_idx][:k]

        # Count how many of the top K belong to the correct folder
        relevant_retrieved = sum(1 for idx in top_indices if master_map.index[idx].split('/')[0] == query_folder)

        # Calculate metrics for this specific query
        precision = relevant_retrieved / k
        recall = relevant_retrieved / total_relevant_in_db if total_relevant_in_db > 0 else 0

        total_precision += precision
        total_recall += recall

    # Calculate overall averages
    avg_precision = total_precision / num_test_queries
    avg_recall = total_recall / num_test_queries

    print(f"\n✅ --- Final Baseline Evaluation Results ---")
    print(f"Average Precision@{k}: {avg_precision:.4f} ({(avg_precision*100):.2f}%)")
    print(f"Average Recall@{k}:    {avg_recall:.4f} ({(avg_recall*100):.2f}%)")
    print(f"-------------------------------------------")

# Run the evaluation!
evaluate_baseline(k=5, num_test_queries=100)
# You can also test with different K values to see how the engine behaves
# evaluate_baseline(k=10, num_test_queries=100)

📊 Running Baseline Evaluation (k=5, queries=100)...

✅ --- Final Baseline Evaluation Results ---
Average Precision@5: 0.7620 (76.20%)
Average Recall@5:    0.0068 (0.68%)
-------------------------------------------
